In [13]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
import seaborn as sns
from pathlib import Path
import plotly.graph_objects as go
pio.renderers.default = "notebook_connected"

In [14]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.imputation import load_time_indexed_csv, impute_loads_by_gap_categories_safe, apply_imputed_columns, missing_summary
from src.time_features import add_calendar_features, get_calendar_feature_columns
from src.lag_features import add_lag_features, add_rolling_features, add_trend_features
from src.dataset_builder import build_multiple_target_datasets, save_target_datasets_parquet

In [15]:
INPUT_PATH = r"C:\Data_analysis\Thesis\Data\02_Preprocessing\LF_data_droped.csv"
SAVE_DIR = Path(r"C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\5_min") # chaneg path when resolution is diff.

In [16]:
df = load_time_indexed_csv(INPUT_PATH)
print(df.shape)
print(df.columns)

(33696, 9)
Index(['BA_Soc', 'BU_TotActPwr_Academy', 'BA_TotActPwr_BESS_AC_Panel1',
       'BA_TotActPwr_BESS_AC_Panel2', 'BU_TotActPwr_SDB_EL_Substation',
       'BU_TotActPwr_Tech_Room', 'PV_WS_AirTemp', 'PV_WS_Radiation',
       'PV_WS_RelHum'],
      dtype='str')


In [17]:
impute_cols = [
    "BA_Soc",
    "BU_TotActPwr_Academy",
    "BA_TotActPwr_BESS_AC_Panel1",
    "BA_TotActPwr_BESS_AC_Panel2",
    "BU_TotActPwr_SDB_EL_Substation",
    "BU_TotActPwr_Tech_Room",
    "PV_WS_AirTemp",
    "PV_WS_Radiation",
    "PV_WS_RelHum",
]

df_imputed_only, impute_report = impute_loads_by_gap_categories_safe(
    df,
    impute_cols,
    freq_minutes=15,
    short_gap_hours=2.0,
    medium_gap_hours=24.0,
    min_history=5,
)

df = apply_imputed_columns(df, df_imputed_only, impute_cols)

print(impute_report)
print(df[impute_cols].isna().sum())

                           column  NaNs_after_cat1  NaN_runs_after_cat1  \
0                          BA_Soc             2514                    4   
1            BU_TotActPwr_Academy              300                    4   
2     BA_TotActPwr_BESS_AC_Panel1              300                    4   
3     BA_TotActPwr_BESS_AC_Panel2              300                    4   
4  BU_TotActPwr_SDB_EL_Substation             7086                    4   
5          BU_TotActPwr_Tech_Room              300                    4   
6                   PV_WS_AirTemp             1326                    4   
7                 PV_WS_Radiation             1326                    4   
8                    PV_WS_RelHum             1326                    4   

   min_run  max_run  runs_cat2_(2h_to_24h)  runs_cat3_(>24h)  
0        9     2228                      2                 2  
1        9      264                      3                 1  
2        9      264                      3                 1

In [18]:
ms = missing_summary(df)
print(ms[ms["missing_count"] > 0])

                                missing_count  missing_pct
BU_TotActPwr_SDB_EL_Substation           1152         3.42


In [19]:
df = add_calendar_features(df, include_extended=True)
calendar_cols = get_calendar_feature_columns(include_extended=True)

In [ ]:
target_cols = [
    "BU_TotActPwr_Academy",
    "BA_TotActPwr_BESS_AC_Panel1",
    "BA_TotActPwr_BESS_AC_Panel2",
    "BU_TotActPwr_Tech_Room",
    "BU_TotActPwr_SDB_EL_Substation",
]

support_cols = [
    "BA_Soc",
    "PV_WS_AirTemp",
    "PV_WS_Radiation",
    "PV_WS_RelHum",
]

lags = [1, 2, 3, 4, 8, 12, 24, 48, 96, 192, 288, 672]
rolling_windows = [4, 12, 24, 48, 96, 288]
std_windows = [12, 24, 48, 96]
trend_pairs = [(1, 4), (1, 12), (1, 96), (96, 192)]

In [21]:
df = add_lag_features(df, target_cols, lags=lags)

df = add_rolling_features(
    df,
    target_cols,
    rolling_windows=rolling_windows,
    add_std_windows=std_windows,
)

df = add_trend_features(
    df,
    target_cols,
    trend_pairs=trend_pairs,
)

In [22]:
datasets = build_multiple_target_datasets(
    df,
    target_cols=target_cols,
    support_cols=support_cols,
    calendar_cols=calendar_cols,
    include_calendar=True,
    include_lags=True,
    include_rolls=True,
    include_trends=True,
    drop_startup=True,
    startup_rows=576,
    drop_target_nan=False,
)

In [23]:
for name, dfx in datasets.items():
    print(name, dfx.shape)

BU_TotActPwr_SDB_EL_Substation (33120, 39)


In [24]:
saved_files = save_target_datasets_parquet(datasets, SAVE_DIR)

for k, v in saved_files.items():
    print(k, v, v.exists())

BU_TotActPwr_SDB_EL_Substation C:\Data_analysis\Thesis\Data\03_Training\Imputed_data\5_min\df_BU_TotActPwr_SDB_EL_Substation.parquet True


# Next Steps: Machine Learning Model Development


## 1. Time-Series Data Splitting
The dataset will be split chronologically into:
- **Training set**
- **Validation set**
- **Test set**

This avoids data leakage and ensures the model is evaluated on future unseen data.



## 2. Regression-Based Models
Since load forecasting is a continuous prediction task, regression models will be used, such as:
- Random Forest
- XGBoost


## 3. Multiple Load Forecasting
Separate models will be trained for different load signals while using a consistent training pipeline.



## 4. Model Evaluation
Performance will be evaluated using:
- **MAE**
- **RMSE**
- **MAPE**

## **without imputation and with imputation and comparision of XGboost**